# Known-structure guided design (CV-selected surrogate) — interactive results

Tier-B window + family-MSA proposal, guided-score scale `lam_ex = lam_em = 1, T = 1`, **2 design
cycles × 3 independent trials per task**, editable positions visited in a **random order**. Everything
feeding this is native to this experiment (see `README.md`):

- **tasks**: 36 scaffold→target pairs per role cohort from `curate_pairs.py`, selected for
  scaffold→target *spectral distance*, and grouped by the **scaffold's** surrogate role into
  72 `S-pool` + 36 `S-test`.
- **surrogate** (guides the search): `cnn-max-d1`, refit by `train_final_surrogate.py` on the
  surrogate train+val pool.
- **oracle** (judges the result, never consulted by the search): this experiment's oracle-sweep
  winner, trained on the separate 80/10/10 oracle split.

### Three things about this notebook's numbers

**S-val is folded in, not dropped.** The deployed surrogate was refit on train **∪** val
(`n_train=515`, *"test (91) never trained on"*), so an S-val scaffold sits inside its training pool
exactly as an S-train scaffold does — the train/val boundary belongs to the sweep's single-split
protocol, not to the model these designs were run against. Reporting three tiers implied a
generalization gradient that does not exist. All 108 tasks are loaded and grouped into the two
conditions that do: **`S-pool`** (72 tasks, S-train + S-val — scaffolds inside the surrogate's
training pool) and **`S-test`** (36 tasks, never seen). Nothing was re-run; this is a regrouping of
the same trajectories.

**"The design" for a task is the surrogate-selected trial.** Each task was searched 3 times, and the
3 trials end up in genuinely different places — mean spread **28 nm** of oracle error, larger than any
effect this experiment set out to measure. Reporting the best trial by *oracle* error would use the
held-out judge to choose, which is not something you can do at design time. So every panel below
reports the trial whose **surrogate** error is lowest at the final cycle — a rule the search is
allowed to apply — and shows the full trial range alongside it. §8 prices the other selection rules,
including the oracle-picking one you cannot have.

**Both arms are here.** `ARM` in the next cell switches between the family-PSSM proposal
(`msa_rand3`, `3.1_design_run_MSA/`) and the ESM-2 masked-LM proposal (`esm2_rand3`,
`3.2_design_run_ESM2/`). Everything else about the two runs is identical, so re-running the notebook
with the other value is the controlled comparison.

In [1]:
import os, sys, glob, json
sys.path.insert(0, os.path.abspath("."))
import numpy as np, pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import design_common as C
import score_traj_surrogate as STS

ARM = "msa_rand3"          # "esm2_rand3" for the ESM-2 masked-LM proposal arm
pio_tmpl = "plotly_white"
# Three pair cohorts, TWO conditions: the deployed surrogate was refit on train + val, so an
# S-val scaffold sits inside its training pool exactly as an S-train scaffold does. All 108 tasks
# are loaded; S-train and S-val are reported as one condition (design_common.COHORT_CONDITION).
COHORTS = {coh: C.CONDITION_LABEL[C.condition(coh)] for coh in C.DEFAULT_COHORTS}
LABELS = [C.CONDITION_LABEL[c] for c in C.CONDITIONS]        # ["S-pool", "S-test"] -- plot order
COLOR = {"S-pool": "#2b6c8f", "S-test": "#f6a200"}
TRIAL_COLOR = ["#2b6c8f", "#c5474b", "#3f8f5c"]
# one dataset entry is literally named "11", which read_csv would otherwise turn into an int
DTYPE = {"scaffold_name": str, "target_name": str}

# ---- every per-task trajectory CSV: one row per (task, trial, round) ----------------
PIPE = STS.pipe_dir(ARM)
frames = []
for coh, label in COHORTS.items():
    for fn in sorted(glob.glob(os.path.join(PIPE, coh, "design_*.csv"))):
        df = pd.read_csv(fn, dtype=DTYPE); df["cohort"] = label
        frames.append(df)
traj = pd.concat(frames, ignore_index=True)

# the surrogate's own view of each round (design_knownstruct.py logs only the oracle), keyed by
# (example, trial, round) -- computed + cached by score_traj_surrogate.py, ~20 s on first call
traj = traj.merge(STS.compute(ARM, verbose=False), on=["example", "trial", "round"], how="left")

numcols = ["trial", "round", "peak_err", "pred_ex", "pred_em", "surr_err", "surr_ex", "surr_em",
           "fam_logp", "ident_to_scaffold", "seq_id_scaf_target", "target_ex", "target_em",
           "scaffold_ex", "scaffold_em"]
traj[numcols] = traj[numcols].apply(pd.to_numeric, errors="coerce")

# fam_logp is a SUM over the window, and windows here run 14-34 positions, so the raw number is
# partly reading window size across tasks -- normalize to per-position. (The CSV's "n_editable"
# column is the count of positions TOUCHED that round, not the window size, so take that from
# design_windows.json.)
NWIN = {nm: w["n_editable"] for nm, w in json.loads(C.WINDOWS_JSON.read_text())["windows"].items()}
traj["n_win"] = traj["scaffold_name"].map(NWIN)
traj["fam_logp_pp"] = traj["fam_logp"] / traj["n_win"]

LAST = int(traj["round"].max())
TRIALS = sorted(traj["trial"].unique())

# ---- per-task summary ---------------------------------------------------------------
# THE DESIGN IS THE SURROGATE-SELECTED TRIAL AT THE FINAL CYCLE. Picking by oracle error -- across
# either trials or cycles -- would consult the held-out judge, which the search cannot do; picking
# by surrogate error is a rule it CAN apply, so that is what sections 1-7 report. trial_min/max
# carry the full range for the error bars, and §8 prices every other selection rule.
def _summarise(g):
    s0 = g[g["round"] == 0].iloc[0]
    f = g[g["round"] == LAST]
    pick = f.loc[f["surr_err"].idxmin()]
    return pd.Series(dict(
        task=s0["example"], cohort=s0["cohort"], pdb=s0["scaffold_pdb"],
        scaffold=s0["scaffold_name"], target=s0["target_name"], seq_id=s0["seq_id_scaf_target"],
        n_win=int(s0["n_win"]), scaf_err=s0["peak_err"], scaf_surr=s0["surr_err"],
        sel_trial=int(pick["trial"]), des_err=pick["peak_err"], des_surr=pick["surr_err"],
        trial_min=f["peak_err"].min(), trial_max=f["peak_err"].max(), trial_mean=f["peak_err"].mean(),
        spread=f["peak_err"].max() - f["peak_err"].min(),
        d_err=pick["peak_err"] - s0["peak_err"],
        id_scaf=pick["ident_to_scaffold"],
        famlogp0=s0["fam_logp_pp"], famlogp=pick["fam_logp_pp"],
        des_ex=pick["pred_ex"], des_em=pick["pred_em"],
        scaf_pred_ex=s0["pred_ex"], scaf_pred_em=s0["pred_em"],
        target_ex=s0["target_ex"], target_em=s0["target_em"]))
summ = pd.DataFrame([_summarise(g) for _, g in traj.groupby("example", sort=False)]).reset_index(drop=True)
summ["improved"] = summ["d_err"] < 0
summ["all_improved"] = summ["trial_max"] < summ["scaf_err"]     # every trial beat the scaffold

n = len(summ)
print(f"arm {ARM} | {n} tasks x {len(TRIALS)} trials | {LAST} design cycles | "
      f"visit order random | S-pool = S-train + S-val")
print(f"design = surrogate-selected trial: improved on the scaffold {int(summ['improved'].sum())}/{n} "
      f"| every trial improved {int(summ['all_improved'].sum())}/{n}")
print(f"  mean oracle err: scaffold {summ.scaf_err.mean():.1f} -> design {summ.des_err.mean():.1f} nm "
      f"(mean of all trials {summ.trial_mean.mean():.1f}, best-of-3 by oracle {summ.trial_min.mean():.1f})")
print(f"  per-task spread across trials: mean {summ.spread.mean():.1f} nm, median "
      f"{summ.spread.median():.1f}, max {summ.spread.max():.1f}")
for label in LABELS:
    s = summ[summ.cohort == label]
    print(f"  {label}: {int(s['improved'].sum())}/{len(s)} improved | scaffold {s.scaf_err.mean():.1f} "
          f"-> design {s.des_err.mean():.1f} nm | spread {s.spread.mean():.1f} nm | id kept {s.id_scaf.mean():.0%}")
summ.sort_values("d_err").head(3)[["task", "cohort", "seq_id", "scaf_err", "des_err", "spread", "sel_trial"]]

arm msa_rand3 | 108 tasks x 3 trials | 2 design cycles | visit order random | S-pool = S-train + S-val
design = surrogate-selected trial: improved on the scaffold 103/108 | every trial improved 91/108
  mean oracle err: scaffold 133.2 -> design 87.2 nm (mean of all trials 94.0, best-of-3 by oracle 80.0)
  per-task spread across trials: mean 28.1 nm, median 25.4, max 82.8
  S-pool: 68/72 improved | scaffold 130.7 -> design 85.6 nm | spread 28.5 nm | id kept 92%
  S-test: 35/36 improved | scaffold 138.1 -> design 90.5 nm | spread 27.4 nm | id kept 92%


,task,cohort,seq_id,scaf_err,des_err,spread,sel_trial
76,HcRed-Bluebonnet2,S-test,0.590,189.77,47.80,24.06,2
100,mRubyFT-mCarmine,S-test,0.697,202.44,83.52,20.63,0
26,mNeptune-mBlueberry1,S-pool,0.540,195.44,87.98,39.61,0


## 1. Did design help? Scaffold vs final-design oracle error

Each point is a task: **x** = the scaffold's distance from the target, **y** = the surrogate-selected
design's. **Below the dashed diagonal = improved.** The vertical whisker spans the 3 trials, so a
point sitting low with a long whisker means the selection rule got lucky rather than the search being
reliable. Log axes handle the far-red outliers.

In [2]:
lim = float(max(summ.scaf_err.max(), summ.trial_max.max())) * 1.15
fig = go.Figure()
for label in LABELS:
    s = summ[summ.cohort == label]
    fig.add_trace(go.Scatter(
        x=s.scaf_err, y=s.des_err, mode="markers", name=label,
        marker=dict(color=COLOR[label], size=9, symbol="circle",
                    line=dict(width=1, color="white")),
        error_y=dict(type="data", symmetric=False,
                     array=s.trial_max - s.des_err, arrayminus=s.des_err - s.trial_min,
                     color=COLOR[label], thickness=1, width=3),
        customdata=np.stack([s.task, s.seq_id, s.id_scaf, s.pdb, s.sel_trial,
                             s.trial_min, s.trial_max, s.spread], axis=-1),
        hovertemplate="<b>%{customdata[0]}</b><br>scaffold %{x:.1f} -> design %{y:.1f} nm"
                      "<br>trial range %{customdata[5]:.1f}-%{customdata[6]:.1f} nm "
                      "(spread %{customdata[7]:.1f})<br>selected trial %{customdata[4]}"
                      "<br>scaf->target id %{customdata[1]:.0%} | id kept %{customdata[2]:.0%}"
                      "<br>PDB %{customdata[3]}<extra></extra>"))
fig.add_trace(go.Scatter(x=[1, lim], y=[1, lim], mode="lines", line=dict(dash="dash", color="#888"),
                         name="no change", hoverinfo="skip"))
fig.update_layout(template=pio_tmpl, width=780, height=640,
                  xaxis=dict(title="scaffold->target oracle error (nm)", type="log"),
                  yaxis=dict(title="design->target oracle error (nm)", type="log"),
                  legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.75)"),
                  title=f"{ARM}: surrogate-selected design vs scaffold "
                        f"({int(summ.improved.sum())}/{len(summ)} improved; whisker = 3 trials)")
fig.show()

## 2. Per-task error change (Δ oracle error)

Sorted; **negative (left) = design improved** the oracle peak error. The bar is the
surrogate-selected trial; the whisker spans what the other trials would have given, so a bar whose
whisker crosses zero is a task where the search *can* help but is not guaranteed to.

In [3]:
b2 = summ.sort_values("d_err")
fig = go.Figure()
for label in LABELS:
    s = b2[b2.cohort == label]
    fig.add_trace(go.Bar(
        x=s.task, y=s.d_err, name=label, marker_color=COLOR[label],
        error_y=dict(type="data", symmetric=False,
                     array=s.trial_max - s.des_err, arrayminus=s.des_err - s.trial_min,
                     color="#777", thickness=1, width=2),
        customdata=np.stack([s.seq_id, s.scaf_err, s.des_err, s.trial_min, s.trial_max,
                             s.sel_trial], axis=-1),
        hovertemplate="<b>%{x}</b><br>Δ %{y:+.1f} nm<br>scaffold %{customdata[1]:.1f} -> "
                      "design %{customdata[2]:.1f} nm<br>trial range %{customdata[3]:.1f}-"
                      "%{customdata[4]:.1f} (selected %{customdata[5]})"
                      "<br>scaf->target id %{customdata[0]:.0%}<extra></extra>"))
fig.add_hline(y=0, line_dash="dash", line_color="#c5474b")
fig.update_layout(template=pio_tmpl, barmode="relative", width=1250, height=560,
                  xaxis=dict(title="", categoryorder="array", categoryarray=list(b2.task),
                             tickangle=-60, tickfont_size=8),
                  yaxis_title="Δ oracle err (nm)  (negative = better)",
                  legend_title_text="cohort",
                  title="Per-task oracle-error change (surrogate-selected design vs scaffold)")
fig.show()

## 3. Per-task trajectory viewer (use the dropdown)

Peak error vs design cycle for one task; **round 0 = the scaffold**, and the three coloured curves are
the three independent trials. Pick a task from the dropdown; click a trial in the legend to hide it.

Each trial is drawn twice:

- **solid, filled markers = oracle** — the held-out judge, the honest number, what every other section
  reports.
- **dotted, open markers = surrogate** — the same sequence scored by the model that *guided* the
  search, recomputed post-hoc by `score_traj_surrogate.py` (the design CSVs log only the oracle).

So the figure carries both sources of variance at once. The **spread between the solid curves** is the
search's own randomness on this task — nothing about the task changed between trials, only the random
stream. The **gap between a trial's solid and dotted curve** is the surrogate↔oracle disagreement, the
model error the search is flying blind on. ★ in the title marks the trial the surrogate selects, which
is the one sections 1-7 report.

In [4]:
tasks = list(summ.sort_values(["cohort", "d_err"])["task"])
NT = len(TRIALS)
PER = 2 * NT                       # oracle + surrogate curve per trial

def _title(tk):
    r = summ[summ.task == tk].iloc[0]
    return (f"{tk}  [{r['cohort']}, {r['seq_id']:.0%} id, PDB {r['pdb']}]<br>"
            f"<sup>scaffold {r['scaf_err']:.1f} nm  |  trials end at "
            f"{r['trial_min']:.1f}-{r['trial_max']:.1f} nm (spread {r['spread']:.1f})  |  "
            f"surrogate selects trial {r['sel_trial']} ★ -> oracle {r['des_err']:.1f} nm</sup>")

fig = go.Figure()
for k, tk in enumerate(tasks):
    sel = int(summ.loc[summ.task == tk, "sel_trial"].iloc[0])
    for tr in TRIALS:
        g = traj[(traj.example == tk) & (traj.trial == tr)].sort_values("round")
        star = " ★" if tr == sel else ""
        fig.add_trace(go.Scatter(
            x=g["round"], y=g["peak_err"], mode="lines+markers", visible=(k == 0),
            name=f"trial {tr}{star}", legendgroup=f"t{tr}",
            line=dict(color=TRIAL_COLOR[tr % len(TRIAL_COLOR)], width=3 if tr == sel else 2),
            marker=dict(size=9),
            hovertemplate=f"trial {tr} · <b>oracle</b><br>round %{{x}}<br>err %{{y:.1f}} nm "
                          "(%{customdata[0]:.0f}/%{customdata[1]:.0f} ex/em)"
                          "<br>fam logp/pos %{customdata[2]:.2f}<br>id->scaf %{customdata[3]:.0%}"
                          "<extra></extra>",
            customdata=np.stack([g["pred_ex"], g["pred_em"], g["fam_logp_pp"],
                                 g["ident_to_scaffold"]], axis=-1)))
        fig.add_trace(go.Scatter(
            x=g["round"], y=g["surr_err"], mode="lines+markers", visible=(k == 0),
            name=f"trial {tr} surrogate", legendgroup=f"t{tr}", showlegend=False,
            line=dict(color=TRIAL_COLOR[tr % len(TRIAL_COLOR)], width=1.6, dash="dot"),
            marker=dict(size=8, symbol="circle-open", line=dict(width=1.6)),
            hovertemplate=f"trial {tr} · <b>surrogate</b><br>round %{{x}}<br>err %{{y:.1f}} nm "
                          "(%{customdata[0]:.0f}/%{customdata[1]:.0f} ex/em)<extra></extra>",
            customdata=np.stack([g["surr_ex"], g["surr_em"]], axis=-1)))

buttons = [dict(label=tk, method="update",
                args=[{"visible": [i // PER == k for i in range(PER * len(tasks))]},
                      {"title": _title(tk)}])
           for k, tk in enumerate(tasks)]
fig.update_layout(
    updatemenus=[dict(buttons=buttons, x=1.02, y=1.0, xanchor="left", direction="down")],
    template=pio_tmpl, width=920, height=580, margin=dict(t=110),
    legend=dict(x=0.98, y=0.98, xanchor="right", yanchor="top", bgcolor="rgba(255,255,255,0.75)"),
    xaxis=dict(title="design cycle (0 = scaffold)", dtick=1),
    yaxis_title="peak error vs target (nm)", title=_title(tasks[0]))
fig.show()

## 4. Movement in the (excitation, emission) plane

For every task: the **scaffold** oracle prediction (circle) is joined to the **surrogate-selected
design** (diamond), which should move toward the **target** (star). Short arrows landing on a star =
success. Only the selected trial is drawn, otherwise the plane is unreadable at 3× the arrows.

In [5]:
fig = go.Figure()
for _, r in summ.iterrows():
    fig.add_trace(go.Scatter(x=[r.scaf_pred_ex, r.des_ex], y=[r.scaf_pred_em, r.des_em],
                             mode="lines", line=dict(color="#cccccc", width=1),
                             showlegend=False, hoverinfo="skip"))

def _pts(ex, em, sym, name, size, opacity=1.0):
    for label in LABELS:
        s = summ[summ.cohort == label]
        fig.add_trace(go.Scatter(
            x=s[ex], y=s[em], mode="markers", name=f"{name} ({label})",
            marker=dict(symbol=sym, size=size, color=COLOR[label], opacity=opacity,
                        line=dict(width=1, color="#444")),
            customdata=np.stack([s.task, s.des_err, s.scaf_err], axis=-1),
            hovertemplate="<b>%{customdata[0]}</b><br>" + name +
                          " %{x:.0f} / %{y:.0f} nm<br>scaffold err %{customdata[2]:.1f} -> "
                          "design %{customdata[1]:.1f} nm<extra></extra>"))
_pts("scaf_pred_ex", "scaf_pred_em", "circle", "scaffold", 8, 0.55)
_pts("des_ex", "des_em", "diamond", "design", 10)
_pts("target_ex", "target_em", "star", "target", 13)
lo = float(min(summ.target_ex.min(), summ.scaf_pred_ex.min(), summ.des_ex.min())) - 20
hi = float(max(summ.target_em.max(), summ.scaf_pred_em.max(), summ.des_em.max())) + 20
fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", line=dict(dash="dot", color="#ddd"),
                         name="ex = em", hoverinfo="skip"))
fig.update_layout(template=pio_tmpl, width=820, height=700,
                  xaxis_title="excitation (nm)", yaxis_title="emission (nm)",
                  legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.7)", font_size=10),
                  title="Scaffold -> surrogate-selected design -> target in the (ex, em) plane")
fig.show()

## 5. Cost of the edits: sequence retained & naturalness

Left: how much identity to the scaffold each design keeps (edits are confined to the chromophore +
Tier-B window). Right: the family-PSSM log-likelihood of scaffold vs design, **per editable position**
— windows here run 14–34 positions, so the raw summed `fam_logp` in the CSVs is partly a readout of
window size and is not comparable across tasks; dividing by the window size fixes that.
**Below the diagonal = the design moved away from the family-typical region** to reach the target.
All 3 trials are shown on both panels, since this is a property of every design produced, not of the
selection rule.

In [6]:
from plotly.subplots import make_subplots
fin = traj[traj["round"] == LAST]
scaf0 = traj[traj["round"] == 0]
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Identity retained to scaffold (all trials)", "Naturalness: family log-likelihood per position"))
for label in LABELS:
    f = fin[fin.cohort == label]
    fig.add_trace(go.Histogram(x=f["ident_to_scaffold"], name=label, marker_color=COLOR[label],
                               opacity=0.75, nbinsx=15, legendgroup=label), 1, 1)
    # scaffold value is identical across a task's trials, so pair each trial's design with it
    pair = f.merge(scaf0[["example", "fam_logp_pp"]].drop_duplicates("example"),
                   on="example", suffixes=("", "_scaf"))
    fig.add_trace(go.Scatter(
        x=pair["fam_logp_pp_scaf"], y=pair["fam_logp_pp"], mode="markers", name=label,
        marker=dict(color=COLOR[label], size=7, opacity=0.75, line=dict(width=0.5, color="white")),
        legendgroup=label, showlegend=False,
        customdata=np.stack([pair["example"], pair["trial"], pair["peak_err"]], axis=-1),
        hovertemplate="<b>%{customdata[0]}</b> (trial %{customdata[1]:.0f})<br>"
                      "fam logp/pos: scaffold %{x:.2f} -> design %{y:.2f}"
                      "<br>oracle err %{customdata[2]:.1f} nm<extra></extra>"), 1, 2)
lo = float(min(fin.fam_logp_pp.min(), scaf0.fam_logp_pp.min())) - 0.2
hi = float(max(fin.fam_logp_pp.max(), scaf0.fam_logp_pp.max())) + 0.2
fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", line=dict(dash="dash", color="#888"),
                         showlegend=False, hoverinfo="skip"), 1, 2)
fig.update_xaxes(title_text="identity to scaffold", tickformat=".0%", row=1, col=1)
fig.update_yaxes(title_text="designs", row=1, col=1)
fig.update_xaxes(title_text="scaffold fam_logp / position", row=1, col=2)
fig.update_yaxes(title_text="design fam_logp / position", row=1, col=2)
fig.update_layout(template=pio_tmpl, width=1120, height=480, barmode="overlay",
                  legend_title_text="cohort",
                  title=f"{ARM}: what the edits cost ({len(fin)} designs = {len(summ)} tasks x {len(TRIALS)} trials)")
fig.show()

print("identity to scaffold / fam_logp per position, all trials:")
for label in LABELS:
    f = fin[fin.cohort == label]
    s = scaf0[scaf0.cohort == label]
    print(f"  {label}: id {f.ident_to_scaffold.mean():.1%} | fam_logp/pos scaffold "
          f"{s.fam_logp_pp.mean():+.2f} -> design {f.fam_logp_pp.mean():+.2f} "
          f"({(f.fam_logp_pp.mean() - s.fam_logp_pp.mean()):+.2f})")

identity to scaffold / fam_logp per position, all trials:
  S-pool: id 92.0% | fam_logp/pos scaffold -1.50 -> design -2.06 (-0.56)
  S-test: id 92.2% | fam_logp/pos scaffold -1.46 -> design -2.10 (-0.64)


## 6. Distribution of the design errors

**Left:** the oracle error of every design produced (3 trials per task), by cohort — the honest picture
of what one run of the search yields. **Right:** the same tasks under three views of the scaffold→design
step: the scaffold, all trials pooled, and the surrogate-selected trial. The gap between the last two is
what the selection rule buys.

In [7]:
print("oracle peak error (nm), scaffold -> design")
for label in LABELS:
    s = summ[summ.cohort == label]
    f = fin[fin.cohort == label]
    print(f"  {label:8}: n={len(s):2d} tasks | scaffold mean {s.scaf_err.mean():5.1f} / median "
          f"{s.scaf_err.median():5.1f}  ->  all trials {f.peak_err.mean():5.1f} / "
          f"{f.peak_err.median():5.1f}   surrogate-selected {s.des_err.mean():5.1f} / {s.des_err.median():5.1f}")
print(f"  {'ALL':8}: n={len(summ):2d} tasks | scaffold {summ.scaf_err.mean():5.1f}  ->  "
      f"all trials {fin.peak_err.mean():5.1f}   selected {summ.des_err.mean():5.1f} nm")

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Every design's oracle error, by cohort", "Scaffold vs all trials vs selected trial"))
for label in LABELS:
    f = fin[fin.cohort == label]
    fig.add_trace(go.Histogram(x=f["peak_err"], name=label, marker_color=COLOR[label],
                               opacity=0.75, nbinsx=22, legendgroup=label), 1, 1)
for x, name, color in [(summ.scaf_err, "scaffold", "#999999"),
                       (fin.peak_err, "all trials", "#7a5ea8"),
                       (summ.des_err, "surrogate-selected", "#2b6c8f")]:
    fig.add_trace(go.Histogram(x=x, name=name, marker_color=color, opacity=0.6, nbinsx=22), 1, 2)
fig.update_xaxes(title_text="oracle peak error (nm)", row=1, col=1)
fig.update_xaxes(title_text="oracle peak error (nm)", row=1, col=2)
fig.update_yaxes(title_text="count", row=1, col=1)
fig.update_layout(template=pio_tmpl, width=1120, height=460, barmode="overlay",
                  title=f"{ARM}: where the designs land")
fig.show()

oracle peak error (nm), scaffold -> design
  S-pool  : n=72 tasks | scaffold mean 130.7 / median 134.6  ->  all trials  92.6 /  91.0   surrogate-selected  85.6 /  87.1
  S-test  : n=36 tasks | scaffold mean 138.1 / median 134.9  ->  all trials  96.9 /  98.0   surrogate-selected  90.5 /  93.5
  ALL     : n=108 tasks | scaffold 133.2  ->  all trials  94.0   selected  87.2 nm


## 7. Change in design error vs. how far the scaffold started from the target

**x** = the **scaffold** oracle error. **y** = the change from design,
`Δ = design err − scaffold err` (negative = improved), for the surrogate-selected trial. The dotted
`y = −x` line is the best possible (error driven to 0). Every trial is plotted faintly behind, so the
correlation can be read against the trial noise rather than through it.

In [8]:
pair = fin.merge(scaf0[["example", "peak_err"]].drop_duplicates("example"),
                 on="example", suffixes=("", "_scaf"))
pair["d"] = pair["peak_err"] - pair["peak_err_scaf"]
rho_sel = summ["d_err"].corr(summ["scaf_err"])
rho_all = pair["d"].corr(pair["peak_err_scaf"])
xmax = float(summ.scaf_err.max()) * 1.08
fig = go.Figure()
fig.add_trace(go.Scatter(x=pair["peak_err_scaf"], y=pair["d"], mode="markers", name="all trials",
                         marker=dict(color="#bbbbbb", size=6, opacity=0.6),
                         customdata=np.stack([pair["example"], pair["trial"]], axis=-1),
                         hovertemplate="%{customdata[0]} (trial %{customdata[1]:.0f})"
                                       "<br>scaffold %{x:.1f} nm, Δ %{y:+.1f} nm<extra></extra>"))
for label in LABELS:
    s = summ[summ.cohort == label]
    fig.add_trace(go.Scatter(
        x=s.scaf_err, y=s.d_err, mode="markers", name=f"selected ({label})",
        marker=dict(color=COLOR[label], size=9, line=dict(width=1, color="white")),
        customdata=np.stack([s.task, s.des_err, s.id_scaf, s.sel_trial], axis=-1),
        hovertemplate="<b>%{customdata[0]}</b> (trial %{customdata[3]:.0f})<br>scaffold %{x:.1f} nm"
                      "<br>Δ %{y:+.1f} -> design %{customdata[1]:.1f} nm"
                      "<br>id kept %{customdata[2]:.0%}<extra></extra>"))
fig.add_hline(y=0, line_dash="dash", line_color="#888")
fig.add_trace(go.Scatter(x=[0, xmax], y=[0, -xmax], mode="lines", name="perfect (err -> 0)",
                         line=dict(dash="dot", color="#c5474b"), hoverinfo="skip"))
fig.update_layout(template=pio_tmpl, width=860, height=600,
                  xaxis_title="scaffold->target oracle error (nm)",
                  yaxis_title="Δ oracle err, design - scaffold (nm)",
                  legend=dict(x=0.02, y=0.06, bgcolor="rgba(255,255,255,0.75)", font_size=10),
                  title=f"Harder starts gain more: rho = {rho_sel:.2f} (selected), "
                        f"{rho_all:.2f} (all {len(pair)} designs)")
fig.show()

## 8. What is actually selectable? (trial variance and cycle non-monotonicity)

With 3 trials × 2 cycles each task produced **6 candidate designs**, and the run is stochastic: the
proposal's top-`k` is sampled at `T = 1` and nothing rejects a worse sequence, so a later cycle is not
a refinement of an earlier one but close to an independent redraw. Two consequences are worth separating,
because only one of them is a result you can act on.

**The trial axis is selectable.** Choosing among trials by *surrogate* error is a rule the search may
apply — the surrogate is its own scorer, freely available at design time. The table below prices it
against the two references that bracket it: what one arbitrary run gives you (no selection), and what
picking by *oracle* error would give (not obtainable, since the oracle is the held-out judge).

**The cycle axis mostly is not.** A per-cycle "we passed through something better" statistic is
`max(0, err₂ − err₁)`, the positive part of a difference whose mean is ≈ 0 — so it is biased positive
even for a search that only diffuses, and it reports noise as lost headroom. The sign test below is the
honest version: if cycle 2 were systematically worse than cycle 1 there would be a consistent sign, and
there is not. What *is* real is that the second cycle buys almost nothing on the mean, which argues for
adding a surrogate-side accept-only-if-better step to the pipeline rather than for reporting headroom.

In [9]:
# ---- the two axes, priced ------------------------------------------------------------
cand = traj[traj["round"] >= 1]        # every candidate design: 3 trials x 2 cycles per task
def _pick(by):
    """Per task, the ORACLE error of the candidate that minimizing `by` would select."""
    idx = cand.groupby("example")[by].idxmin()
    return cand.loc[idx].set_index("example")["peak_err"]

pol = pd.DataFrame({
    "scaffold (no design)": summ.set_index("task").scaf_err,
    "one arbitrary trial (trial 0, final cycle)":
        fin[fin.trial == 0].set_index("example").peak_err,
    "mean over the 3 trials (expected single run)": summ.set_index("task").trial_mean,
    "surrogate-selected TRIAL, final cycle  [§1-7]": summ.set_index("task").des_err,
    "surrogate-selected TRIAL x CYCLE": _pick("surr_err"),
    "oracle-selected trial, final cycle  (not obtainable)": summ.set_index("task").trial_min,
    "oracle-selected trial x cycle  (not obtainable)": _pick("peak_err"),
})
print(f"{ARM}: mean ORACLE error (nm) of the design each selection rule returns, n={len(pol)} tasks\n")
for c in pol.columns:
    tag = "  <-- what sections 1-7 report" if "§1-7" in c else ""
    print(f"  {pol[c].mean():6.1f}   {c}{tag}")
imp = pol["surrogate-selected TRIAL, final cycle  [§1-7]"].mean()
none, best = pol["mean over the 3 trials (expected single run)"].mean(), pol["oracle-selected trial x cycle  (not obtainable)"].mean()
print(f"\nthe surrogate recovers {none - imp:.1f} nm of the {none - best:.1f} nm that oracle-peeking "
      f"would ({(none - imp) / (none - best):.0%}) -- selection across trials is the part of this "
      f"the search can legitimately do.")

# ---- is the cycle axis non-monotonic in any systematic way? --------------------------
w = traj.pivot_table(index=["example", "trial"], columns="round", values="peak_err")
d = w[LAST] - w[LAST - 1]
print(f"\ncycle {LAST-1} -> {LAST}, over all {len(d)} trajectories:")
print(f"  worse {int((d > 0).sum())} / better {int((d < 0).sum())} | mean {d.mean():+.1f} nm "
      f"(sd {d.std():.1f}) | median {d.median():+.1f}  -> no systematic drift")
print(f"  E|Δ| = {d.abs().mean():.1f} nm, so the naive 'headroom lost' statistic max(0, Δ) averages "
      f"{d.clip(lower=0).mean():.1f} nm by construction, even at zero drift")
print(f"  mean error by cycle: " + " | ".join(f"round {r} {traj[traj['round']==r].peak_err.mean():.1f}"
                                              for r in range(LAST + 1)))

# ---- and the picture: trial range per task, with both selections marked --------------
b3 = summ.sort_values("spread", ascending=False)
fig = go.Figure()
fig.add_trace(go.Bar(x=b3.task, y=b3.trial_max - b3.trial_min, base=b3.trial_min,
                     marker=dict(color="#dddddd", line=dict(width=0)), name="trial range",
                     hovertemplate="<b>%{x}</b><br>trials span %{base:.1f}-%{y:.1f}<extra></extra>"))
for label in LABELS:
    s = b3[b3.cohort == label]
    fig.add_trace(go.Scatter(x=s.task, y=s.des_err, mode="markers", name=f"surrogate-selected ({label})",
                             marker=dict(color=COLOR[label], size=8, symbol="diamond"),
                             customdata=np.stack([s.sel_trial, s.spread], axis=-1),
                             hovertemplate="<b>%{x}</b><br>selected trial %{customdata[0]:.0f} -> "
                                           "%{y:.1f} nm (spread %{customdata[1]:.1f})<extra></extra>"))
fig.add_trace(go.Scatter(x=b3.task, y=b3.trial_min, mode="markers", name="oracle-best (not obtainable)",
                         marker=dict(color="#c5474b", size=6, symbol="x"),
                         hovertemplate="<b>%{x}</b><br>oracle-best trial %{y:.1f} nm<extra></extra>"))
fig.update_layout(template=pio_tmpl, width=1250, height=560,
                  xaxis=dict(title="", categoryorder="array", categoryarray=list(b3.task),
                             tickangle=-60, tickfont_size=8),
                  yaxis_title="final-cycle oracle error (nm)",
                  legend=dict(x=0.7, y=0.98, bgcolor="rgba(255,255,255,0.8)", font_size=10),
                  title="Trial variance dominates: each bar spans one task's 3 trials, "
                        "sorted by spread")
fig.show()

msa_rand3: mean ORACLE error (nm) of the design each selection rule returns, n=108 tasks

   133.2   scaffold (no design)
    95.2   one arbitrary trial (trial 0, final cycle)
    94.0   mean over the 3 trials (expected single run)
    87.2   surrogate-selected TRIAL, final cycle  [§1-7]  <-- what sections 1-7 report
    84.9   surrogate-selected TRIAL x CYCLE
    80.0   oracle-selected trial, final cycle  (not obtainable)
    72.6   oracle-selected trial x cycle  (not obtainable)

the surrogate recovers 6.8 nm of the 21.5 nm that oracle-peeking would (32%) -- selection across trials is the part of this the search can legitimately do.

cycle 1 -> 2, over all 324 trajectories:
  worse 145 / better 179 | mean -1.6 nm (sd 24.9) | median -1.9  -> no systematic drift
  E|Δ| = 18.9 nm, so the naive 'headroom lost' statistic max(0, Δ) averages 8.6 nm by construction, even at zero drift
  mean error by cycle: round 0 133.2 | round 1 95.6 | round 2 94.0
